In [ ]:
import json
import subprocess
import time
import statistics

In [ ]:
# query anchors from the city list. This code can be skipped if you have city_ping_results.json file.

anchors_by_city = {}

with open('30-largest-cities-by-population-2026.json', 'r') as file_cities:
    data = json.load(file_cities)
    for entry in data:
        anchors = []

        city = entry['city']
        city_encoded = city.replace(' ', '%20')
        url = f'https://atlas.ripe.net/api/v2/anchors?search={city_encoded}'

        cp = subprocess.run(
            ['curl', '-s', url],
            capture_output=True,
            text=True,
            check=True
        )
        curl_result = json.loads(cp.stdout.strip())
        for anchor_entry in curl_result['results']:
            if not 'ip_v4' in anchor_entry:
                continue
            if anchor_entry['ip_v4'] in [None, '']:
                continue
            if anchor_entry['is_disabled'] == True:
                continue
            if anchor_entry['date_decommissioned'] != None:
                continue

            anchors.append(anchor_entry)    

        anchors_by_city[city] = anchors
        print(f'{city}: {len(anchors)}')


In [ ]:
city_by_anchor_probe = {}

for city, anchors in anchors_by_city.items():
    for anchor in anchors:
        city_by_anchor_probe[anchor['probe']] = city

In [ ]:
city_measurements = {} #dict[city]dict[fqdn]measurement

for city, anchors in anchors_by_city.items():
    anchor_measurements = {} #key: fqdn

    for anchor in anchors:
        measurements = []

        #print(anchor)
        url = f'https://atlas.ripe.net/api/v2/measurements/?target={anchor['fqdn']}&tags=anchoring,mesh&type=ping&status=Ongoing&is_public=True'
        cp = subprocess.run(
            ['curl', '-s', url],
            capture_output=True,
            text=True,
            check=True
        )
        curl_result = json.loads(cp.stdout.strip())

        for measurement in curl_result['results']:
            if ':' in measurement['target_ip']: #ipv6
                continue

            measurements.append(measurement)

        anchor_measurements[anchor['fqdn']] = measurements
        print(f'{anchor['fqdn']}: {len(measurements)}')
    
    city_measurements[city] = anchor_measurements


In [ ]:
time_stop = int(time.time())
time_start = time_stop - 5 * 60 * 60 #an hour
print(f'start:{time_start}, stop:{time_stop}')

In [ ]:
# for each measurement, we query every other city anchor's probe
all_probes_str = ','.join(map(str, city_by_anchor_probe.keys()))

In [ ]:
city_ping_results = {} # dict['city|target']dict['probe']list[rtt]

for city, city_entry in city_measurements.items():
    for fqdn, measurements in city_entry.items():
        for measurement in measurements:
            city_msm_id = f'{city}|{measurement['target']}'
            print(city_msm_id)

            ping_results = {}
            for probe_id in city_by_anchor_probe.keys():
                ping_results[probe_id] = []

            url = f'{measurement['result']}?probe_ids={all_probes_str}&start={time_start}&stop={time_stop}&format=json'
            cp = subprocess.run(
                ['curl', '-s', url],
                capture_output=True,
                text=True,
                check=True
            )
            curl_result = json.loads(cp.stdout.strip())

            for result_entry in curl_result:
                #print(result_entry)
                for trips in result_entry['result']:
                    rtt = trips.get('rtt')
                    if rtt == None:
                        continue
                    try:
                        rtt_float = float(rtt)
                        ping_results[result_entry['prb_id']].append(rtt_float)
                    finally:
                        pass

            city_ping_results[city_msm_id] = ping_results
            

In [ ]:
with open('./city_ping_results.json', 'w') as f_city_ping_result_dump:
    json.dump(city_ping_results, f_city_ping_result_dump)

In [ ]:
######## This block should be skipped if you already ran the previous codes in this session. ########
city_ping_results = {}

with open('./city_ping_results.json', 'r') as f_city_ping_result_dump:
    city_ping_results = json.load(f_city_ping_result_dump)
#####################################################################################################

In [ ]:
def join_city_name(city1:str, city2:str) -> str:
    return f'{city1}|{city2}' if city1 < city2 else f'{city2}|{city1}'

inter_city_rtts = {}
for city1 in city_by_anchor_probe.values():
    for city2 in city_by_anchor_probe.values():
        inter_city_rtts[join_city_name(city1, city2)] = []

for city_anchor_name, city_entry in city_ping_results.items():
    for probe_id, rtts in city_entry.items():
        if len(rtts) == 0:
            continue

        probe_city = city_by_anchor_probe[int(probe_id)]
        cities = join_city_name(city_anchor_name.split('|')[0], probe_city)

        print(f'{city_anchor_name}:{probe_id}')
        inter_city_rtts[cities].append(rtts)

In [ ]:
inter_city_rtt_between_best_probes = {}
for cities, rtts in inter_city_rtts.items():
    if len(rtts) == 0:
        continue

    print(f'{cities}')
    inter_city_rtt_between_best_probes[cities] = min(rtts, key=min)

In [ ]:
with open('./inter_city_rtt_between_best_probes.json', 'w') as f_inter_city_rtt_between_best_probes_dump:
    json.dump(inter_city_rtt_between_best_probes, f_inter_city_rtt_between_best_probes_dump)

In [ ]:
valid_cities = {}
for inter_city in inter_city_rtt_between_best_probes.keys():
    split = inter_city.split('|')
    valid_cities[split[0]] = 0
    valid_cities[split[1]] = 0

valid_cities = list(valid_cities.keys())


In [ ]:
network_stats = {}

for inter_city, rtts in inter_city_rtt_between_best_probes.items():
    split = inter_city.split('|')
    if split[0] == split[1]:
        continue
    network_stats[inter_city] = {'mean': sum(rtts) / len(rtts), 'stddev': statistics.stdev(rtts)}

In [ ]:
with open('./network_stats.json', 'w') as f_network_stats:
    json.dump(network_stats, f_network_stats, indent=4)

In [ ]:
print(valid_cities)
print(len(network_stats))